# Aura CosyVoice Colab Starter

This notebook is the first GPU workbench for Aura. It starts with setup and zero-shot synthesis before fine-tuning.

Run the cells from top to bottom. If a cell fails, stop and read the error before continuing.

## 0. Before You Start

In Colab, choose:

`Runtime -> Change runtime type -> T4 GPU` or better.

This notebook assumes you have a folder in Google Drive for Aura assets. You can upload a few clean WAV clips manually at first.

In [ ]:
# Check the GPU. If this says no GPU, change the Colab runtime type.
!nvidia-smi

## 1. Mount Google Drive

Use Drive as the persistent place for uploaded voice clips, generated audio, and exported Vault assets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

AURA_ROOT = Path('/content/drive/MyDrive/aura')
RAW_CLIPS = AURA_ROOT / 'raw_clips'
CLEAN_CLIPS = AURA_ROOT / 'clean_clips'
OUTPUTS = AURA_ROOT / 'outputs'
VAULT = AURA_ROOT / 'vault'

for path in [RAW_CLIPS, CLEAN_CLIPS, OUTPUTS, VAULT]:
    path.mkdir(parents=True, exist_ok=True)

print('Aura Drive folder:', AURA_ROOT)
print('Upload source clips to:', RAW_CLIPS)

## 2. Install System Packages

CosyVoice needs audio tools. This can take a few minutes.

In [ ]:
!apt-get update -y
!apt-get install -y sox libsox-dev ffmpeg git git-lfs

## 3. Clone CosyVoice

This pulls the official Apache-2.0 CosyVoice repo and its submodules.

In [ ]:
%cd /content
!test -d CosyVoice || git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd /content/CosyVoice
!git submodule update --init --recursive

## 4. Install Python Dependencies

If Colab reports dependency conflicts, copy the error into chat and stop here.

In [ ]:
%cd /content/CosyVoice
!pip install -q -r requirements.txt
!pip install -q huggingface_hub modelscope soundfile librosa

## 5. Download a Pretrained CosyVoice Model

Start with CosyVoice2-0.5B. It is a practical first target for zero-shot testing and later fine-tuning.

In [ ]:
%cd /content/CosyVoice
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='FunAudioLLM/CosyVoice2-0.5B',
    local_dir='pretrained_models/CosyVoice2-0.5B',
    local_dir_use_symlinks=False,
)

print('Downloaded CosyVoice2-0.5B')

## 6. Upload or Copy a Reference Clip

For the first test, use one clean WAV clip around 5-15 seconds. It should contain only the target speaker, minimal noise, and no music.

Upload it into `/content/drive/MyDrive/aura/raw_clips`, then run the next cell.

In [ ]:
clips = sorted(RAW_CLIPS.glob('*'))
print('Found clips:')
for clip in clips:
    print(' -', clip.name)

assert clips, f'No clips found. Upload a WAV/MP3/M4A file to {RAW_CLIPS}'

## 7. Normalize the Reference Clip

This converts the first uploaded clip to 24kHz mono WAV. Later we will make this a full dataset cleaning step.

In [ ]:
import subprocess

source_clip = clips[0]
reference_wav = CLEAN_CLIPS / 'reference_24k_mono.wav'

subprocess.run([
    'ffmpeg', '-y', '-i', str(source_clip),
    '-ar', '24000', '-ac', '1',
    str(reference_wav),
], check=True)

print('Reference WAV:', reference_wav)

## 8. Write the Reference Transcript

Replace the text below with the exact words spoken in your reference clip. This matters a lot.

In [ ]:
reference_text = "REPLACE THIS WITH THE EXACT TRANSCRIPT OF THE REFERENCE CLIP"

if reference_text.startswith('REPLACE THIS'):
    raise ValueError('Edit reference_text before continuing.')

print(reference_text)

## 9. First Zero-Shot Synthesis Test

This is not fine-tuning yet. It checks whether the base model can use your reference voice.

In [ ]:
%cd /content/CosyVoice

import sys
sys.path.append('/content/CosyVoice')

from cosyvoice.cli.cosyvoice import CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B')

prompt_speech = load_wav(str(reference_wav), 16000)
target_text = "Please, just stop. If they find out we are down here, I do not know what they will do to us."

results = cosyvoice.inference_zero_shot(
    target_text,
    reference_text,
    prompt_speech,
    stream=False,
)

output_path = OUTPUTS / 'zero_shot_test.wav'
for result in results:
    torchaudio.save(str(output_path), result['tts_speech'], cosyvoice.sample_rate)

print('Saved:', output_path)

In [ ]:
from IPython.display import Audio, display
display(Audio(str(output_path)))

## 10. Decide Whether to Fine-Tune

Only fine-tune after you have listened to the zero-shot output.

Use this decision rule:

- If speaker identity is close and emotion is acceptable, improve the Director prompts first.
- If identity is unstable across passages, prepare a speaker dataset and fine-tune.
- If emotion is weak, collect balanced clips with emotion labels before fine-tuning.

For the first fine-tune, target one speaker and one narrow emotional domain, such as `M1_fear`.

## 11. Fine-Tuning Prep Checklist

Do not start training until these are true:

- You have permission to use every clip.
- Each clip is clean speech only.
- Each clip has an exact transcript.
- Clips are normalized to mono WAV.
- The dataset has a stable speaker id.
- You can regenerate the dataset from source files.

Next notebook: convert Aura metadata into CosyVoice training files: `wav.scp`, `text`, `utt2spk`, `spk2utt`, embeddings, speech tokens, and parquet lists.